# Student Performance Analysis and Prediction
**Minor Project — Boosting**  
**Submitted by:** KOYYANA PARISRITH REDDY

## 1. Problem Statement
Student performance is influenced by multiple factors, and understanding score patterns can help educators identify where support may be useful. This project explores exam scores and compares simple regression models to predict a student's **math score** from available demographic and preparation features.

## 2. Objectives
- Load a Kaggle dataset and inspect its structure.
- Check data quality: missing values, duplicate rows, types, and unusual values.
- Clean and prepare categorical and numerical features.
- Create at least four meaningful visualizations and write observations.
- Train and evaluate regression models with suitable metrics.
- Explain limitations and future improvements.

## 3. Dataset
**Dataset:** Students Performance in Exams  
**Kaggle link:** https://www.kaggle.com/datasets/spscientist/students-performance-in-exams  
The commonly distributed file contains students' math, reading, and writing scores plus categorical fields such as gender, race/ethnicity group, parental education, lunch type, and test preparation course. Verify the dataset's current description and license on Kaggle before submission. The notebook downloads the CSV from a public raw mirror for convenience; if the link fails, upload `StudentsPerformance.csv` from the Kaggle download when prompted.

In [ ]:
# Install/import libraries used in this notebook
import os
import urllib.request
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

## 4. Load the dataset
The cell first looks for a local CSV. If none exists, it tries a public raw mirror. In Colab, you can also upload the Kaggle CSV.

In [ ]:
DATA_FILE = "StudentsPerformance.csv"
RAW_URL = "https://raw.githubusercontent.com/plotly/datasets/master/StudentsPerformance.csv"

if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
else:
    try:
        urllib.request.urlretrieve(RAW_URL, DATA_FILE)
        df = pd.read_csv(DATA_FILE)
        print("Loaded dataset from public raw mirror.")
    except Exception as exc:
        print("Automatic download failed:", exc)
        try:
            from google.colab import files
            uploaded = files.upload()
            csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
            if not csv_files:
                raise FileNotFoundError("Please upload the Kaggle StudentsPerformance.csv file.")
            df = pd.read_csv(csv_files[0])
        except ImportError:
            raise FileNotFoundError(
                "Place StudentsPerformance.csv in the same folder as this notebook, then rerun."
            )

# Normalize column names so later cells use consistent names.
df.columns = (
    df.columns.str.strip().str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)
print("Dataset shape:", df.shape)
display(df.head())

## 5. Understand the data
Each row represents one student's exam record. The target is `math_score`; the remaining fields are candidate predictors. The notebook checks actual columns rather than assuming every field is present.

In [ ]:
display(df.info())
display(df.describe(include="all").T)
print("Column names:", df.columns.tolist())
target = "math_score"
if target not in df.columns:
    raise ValueError(f"Expected target column '{target}'. Check that you uploaded the correct dataset.")

## 6. Data quality checks

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = int(df.duplicated().sum())
print("Missing values by column:")
display(missing.to_frame("missing_count"))
print("Duplicate rows:", duplicates)

# Check whether numeric exam scores fall within the expected 0–100 range.
score_cols = [c for c in ["math_score", "reading_score", "writing_score"] if c in df.columns]
for col in score_cols:
    numeric = pd.to_numeric(df[col], errors="coerce")
    outside = ((numeric < 0) | (numeric > 100)).sum()
    print(f"{col}: values outside 0–100 =", int(outside))

## 7. Cleaning and preparation
Decisions: remove exact duplicate rows; convert score columns to numeric; treat invalid score values outside 0–100 as missing; impute numeric missing values with the median and categorical missing values with the most frequent category inside the modeling pipeline. This keeps preprocessing fitted only on the training split and helps prevent data leakage.

In [ ]:
df_clean = df.drop_duplicates().copy()

for col in score_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
    df_clean.loc[~df_clean[col].between(0, 100), col] = np.nan

# Rows without the target cannot be used for supervised training.
df_clean = df_clean.dropna(subset=[target]).copy()

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Rows removed as exact duplicates:", len(df) - len(df.drop_duplicates()))
print("Remaining missing values:")
display(df_clean.isna().sum().to_frame("missing_count"))

## 8. Exploratory Data Analysis (EDA)
Each visualization includes a simple observation. Interpret the actual plots produced by your run; the observations are prompts to compare with the displayed evidence.

In [ ]:
# Visualization 1: Distribution of math scores
plt.figure(figsize=(8, 5))
plt.hist(df_clean[target].dropna(), bins=15, edgecolor="black")
plt.title("Distribution of Math Exam Scores")
plt.xlabel("Math score")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()
print("Observation: Use the histogram to describe the score range, most common interval, and whether the distribution looks symmetric or skewed.")

In [ ]:
# Visualization 2: Average score by test preparation course, if available
if "test_preparation_course" in df_clean.columns:
    prep_means = df_clean.groupby("test_preparation_course")[score_cols].mean(numeric_only=True)
    prep_means.plot(kind="bar", figsize=(9, 5))
    plt.title("Average Exam Scores by Test Preparation Course")
    plt.ylabel("Average score")
    plt.xlabel("Test preparation course")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    display(prep_means.round(2))
    print("Observation: Compare the average scores for students who completed preparation and those who did not; this is an association, not proof of causation.")
else:
    print("Column 'test_preparation_course' not found; skip this chart.")

In [ ]:
# Visualization 3: Average math score by lunch type, if available
if "lunch" in df_clean.columns:
    lunch_means = df_clean.groupby("lunch")[target].mean().sort_values(ascending=False)
    lunch_means.plot(kind="bar", figsize=(7, 5), edgecolor="black")
    plt.title("Average Math Score by Lunch Type")
    plt.ylabel("Average math score")
    plt.xlabel("Lunch category")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    display(lunch_means.round(2))
    print("Observation: Compare the group averages and avoid assuming lunch category itself causes score differences.")
else:
    print("Column 'lunch' not found; skip this chart.")

In [ ]:
# Visualization 4: Relationship between reading and writing scores, if present
if {"reading_score", "writing_score"}.issubset(df_clean.columns):
    plt.figure(figsize=(7, 5))
    plt.scatter(df_clean["reading_score"], df_clean["writing_score"], alpha=0.65)
    plt.title("Reading Score vs Writing Score")
    plt.xlabel("Reading score")
    plt.ylabel("Writing score")
    plt.tight_layout()
    plt.show()
    print("Correlation:", round(df_clean[["reading_score", "writing_score"]].corr().iloc[0, 1], 3))
    print("Observation: A clear upward pattern indicates that reading and writing scores tend to move together.")
else:
    print("Reading/writing score columns not both present; skip this chart.")

In [ ]:
# Optional Visualization 5: Correlation heatmap using Matplotlib only
numeric_df = df_clean.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr()
    plt.figure(figsize=(7, 5))
    plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Correlation Between Numeric Variables")
    plt.tight_layout()
    plt.show()
    display(corr.round(2))

## 9. Modeling approach
This is a **regression** task because the target, math score, is numeric. We compare a simple baseline (predicting the training-set mean) with Linear Regression and Ridge Regression. Categorical columns are one-hot encoded; numeric predictors are median-imputed and scaled. To avoid target leakage, `math_score` is excluded from the input features.

In [ ]:
# Exclude the target and any columns that are entirely empty.
X = df_clean.drop(columns=[target]).copy()
y = df_clean[target].astype(float).copy()
X = X.dropna(axis=1, how="all")

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0)
}
results = []
fitted_models = {}

for name, estimator in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": rmse,
        "R2": r2_score(y_test, pred)
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df.round(3))
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
print("Lowest-RMSE model:", best_name)

## 10. Evaluation and interpretation
- **MAE** is the average absolute prediction error in score points; lower is better.
- **RMSE** penalizes larger errors more strongly; lower is better.
- **R²** compares the model with predicting the test-set mean; values closer to 1 indicate better fit, while negative values indicate performance worse than that baseline on this test set.

Use the printed metric table as the actual result. A model with lower error is preferable on this split, but the result is not proof that it will generalize equally well to every school or population.

In [ ]:
# Visualization 6: Actual vs predicted values for the best model
best_pred = best_model.predict(X_test)
plt.figure(figsize=(7, 5))
plt.scatter(y_test, best_pred, alpha=0.7)
low = min(float(y_test.min()), float(np.min(best_pred)))
high = max(float(y_test.max()), float(np.max(best_pred)))
plt.plot([low, high], [low, high], linestyle="--")
plt.title(f"Actual vs Predicted Math Scores — {best_name}")
plt.xlabel("Actual math score")
plt.ylabel("Predicted math score")
plt.tight_layout()
plt.show()
print("Observation: Points closer to the dashed diagonal represent more accurate predictions.")

In [ ]:
# Inspect a few predictions without exposing any personal identifiers (dataset has no names).
comparison = pd.DataFrame({
    "Actual math score": y_test,
    "Predicted math score": best_pred,
    "Absolute error": np.abs(y_test.to_numpy() - best_pred)
}).reset_index(drop=True)
display(comparison.head(10).round(2))
print("Mean absolute error:", round(mean_absolute_error(y_test, best_pred), 2), "score points")

## 11. Limitations
- The dataset is a limited snapshot and may not represent all schools, locations, or student populations.
- The available fields do not capture every influence on performance (for example, attendance, teaching quality, or study time).
- Relationships between group categories and scores should not be interpreted as causal.
- Results depend on the random train/test split; cross-validation and external data would provide stronger evidence.
- Predictions should support learning and discussion, not be used to label or limit individual students.

## 12. Conclusion
This project follows an end-to-end workflow: dataset loading, data-quality checks, cleaning, exploratory visualization, regression modeling, and evaluation. The final conclusion should be written after running all cells: name the model with the lowest RMSE, report its MAE/RMSE/R², and describe two patterns visible in the charts. Do not invent metric values before running the notebook.

## 13. References
- Kaggle dataset page: https://www.kaggle.com/datasets/spscientist/students-performance-in-exams
- pandas documentation: https://pandas.pydata.org/docs/
- scikit-learn documentation: https://scikit-learn.org/stable/
- Matplotlib documentation: https://matplotlib.org/stable/